# Lesson 2.0: How LLMs Work - A Prompt Engineer's Guide

**Duration:** 30 minutes  
**Level:** Beginner  

---

## Learning Objectives

By the end of this lesson, you will:
- Understand LLMs as next-token probability systems
- Know how tokenization affects prompts and costs
- Grasp embeddings and semantic similarity
- Understand autoregressive generation
- Master key API parameters (temperature, max_tokens)

---

## Setup & Installation

First, let's install the required packages and set up our environment.

In [1]:
# Install required packages
!pip install google-generativeai langchain-google-genai python-dotenv -q

In [ ]:
import os
os.environ['GEMINI_API_KEY'] = 'key here'

In [2]:
from google import genai

# The client automatically picks up the API key from the GEMINI_API_KEY environment variable.
client = genai.Client()

print("List of all available models:")
for model in client.models.list():
    # Print basic model information
    print(f"* Name: {model.name}")
    print(f"  Description: {model.description}")

List of all available models:
* Name: models/gemini-2.5-flash
  Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
* Name: models/gemini-2.5-pro
  Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
* Name: models/gemini-2.0-flash
  Description: Gemini 2.0 Flash
* Name: models/gemini-2.0-flash-001
  Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
* Name: models/gemini-2.0-flash-exp-image-generation
  Description: Gemini 2.0 Flash (Image Generation) Experimental
* Name: models/gemini-2.0-flash-lite-001
  Description: Stable version of Gemini 2.0 Flash-Lite
* Name: models/gemini-2.0-flash-lite
  Description: Gemini 2.0 Flash-Lite
* Name: models/gemini-exp-1206
  Description: Experimental release (March 25th, 2025) of Gemini 2.5 Pro
* Name: models/gemini-2.5-flash-preview-tts
  Descri

In [3]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=f"Explain AI Agents in 10 words and provide the 5 use cases",       
)

print('Response ', response.text)

Response  **AI Agents: Autonomous entities that perceive, reason, and act in environments.**

**Use Cases:**

1.  Customer service chatbots.
2.  Personalized recommendation systems.
3.  Autonomous driving vehicles.
4.  Smart home automation.
5.  Financial trading bots.



---

## 1. LLMs as Probability Machines

### Understanding Next-Token Prediction

At their core, LLMs predict: **"What is the most likely next token given the previous tokens?"**

This is why:
- Prompt wording matters (different words = different probability distributions)
- Context is crucial (more context = better predictions)
- Examples work (they establish patterns the model should follow)

### Industry Example: Product Description Generation

Let's see how the model completes product descriptions based on patterns.

In [4]:
# Using Gemini Direct API
from google.genai import types
from google import genai

client = genai.Client()
# model = genai.GenerativeModel('gemini-3.0-flash')

# Product description completion - watching probability in action
incomplete_descriptions = [
    "This premium wireless headphone features",
    "Our organic coffee is sourced from",
    "The ergonomic office chair provides",
]

print("PRODUCT DESCRIPTION COMPLETION")
print("="*60)
print("Watch how the model predicts natural continuations...\n")

for desc in incomplete_descriptions:

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=f"Complete this product description in one sentence: {desc}",     
    )

    print(f"Start: \"{desc}...\"")
    print(f"Completed: {response.text}")
    print("-"*60)

PRODUCT DESCRIPTION COMPLETION
Watch how the model predicts natural continuations...

Start: "This premium wireless headphone features..."
Completed: This premium wireless headphone features **crystal-clear audio, a comfortable ergonomic design, and long-lasting battery life for an immersive listening experience.**

------------------------------------------------------------
Start: "Our organic coffee is sourced from..."
Completed: Our organic coffee is sourced from high-altitude, shade-grown farms, ensuring a rich, flavorful cup with a sustainable impact.

------------------------------------------------------------
Start: "The ergonomic office chair provides..."
Completed: The ergonomic office chair provides exceptional support and comfort for long hours of work, promoting good posture and reducing strain.

------------------------------------------------------------


### Using LangChain with Gemini

Now let's do the same thing using LangChain, which provides a more structured approach.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize LangChain with Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
)

print("LANGCHAIN + GEMINI: Product Description Generation")
print("="*60)

# Using message structure

response = llm.invoke("You are a professional product copywriter. Complete product descriptions concisely. Complete this: 'This premium wireless headphone features'")
print(f"Response: {response.content}")

LANGCHAIN + GEMINI: Product Description Generation
Response: Okay, here are a few options for completing "This premium wireless headphone features," depending on the target audience and key features:

**Option 1 (Focus on sound quality):**

"This premium wireless headphone features **crystal-clear audio, deep bass, and active noise cancellation for an immersive listening experience.**"

**Option 2 (Focus on comfort and style):**

"This premium wireless headphone features **a luxurious, ergonomic design, plush earcups, and a sleek profile for all-day comfort and effortless style.**"

**Option 3 (Focus on technology and convenience):**

"This premium wireless headphone features **advanced Bluetooth 5.3 connectivity, a long-lasting battery, and intuitive touch controls for seamless, on-the-go listening.**"

**Option 4 (More concise, all-around):**

"This premium wireless headphone features **exceptional sound quality, comfortable all-day wear, and seamless wireless connectivity.**"

**Opt

---

## 2. Tokenization: The Currency of LLMs

### What are Tokens?

LLMs don't read text character by character or word by word. Instead, they break text into **tokens** - subword units that represent common character sequences.

**Key Facts:**
- Tokens are the "currency" - you pay per token
- Context window limits are in tokens, not words
- Different languages tokenize differently (less efficient for non-English)

### Deep Dive: Understanding Tokens with Tiktoken

Now let's use **tiktoken** (OpenAI's tokenizer library) to see exactly how text gets broken into tokens. This gives us a concrete understanding of tokenization.

**Note:** Different models use different tokenizers:
- OpenAI GPT models use tiktoken (cl100k_base for GPT-4, GPT-3.5-turbo)
- Google Gemini uses SentencePiece
- The concepts remain the same across tokenizers

In [7]:
# Install tiktoken
!pip install tiktoken -q

In [8]:
import tiktoken

# Get the tokenizer used by GPT-4 and GPT-3.5-turbo
encoding = tiktoken.get_encoding("cl100k_base")

# You can also get encoding for a specific model:
# encoding = tiktoken.encoding_for_model("gpt-4")

def visualize_tokens(text: str, encoding=encoding):
    """Visualize how text is broken into tokens."""
    tokens = encoding.encode(text)
    token_strings = [encoding.decode([t]) for t in tokens]
    
    print(f"Text: \"{text}\"")
    print(f"Token count: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Tokens: {token_strings}")
    print(f"Characters per token: {len(text)/len(tokens):.2f}")
    print()
    return tokens

# Example 1: Simple English text
print("=" * 70)
print("TOKENIZATION EXAMPLES")
print("=" * 70)

print("\n1️⃣ Simple English sentence:")
visualize_tokens("Hello, how are you today?")

print("2️⃣ Technical/programming text:")
visualize_tokens("def calculate_total(items): return sum(items)")

print("3️⃣ Numbers and special characters:")
visualize_tokens("Price: $1,234.56 (20% off!)")

TOKENIZATION EXAMPLES

1️⃣ Simple English sentence:
Text: "Hello, how are you today?"
Token count: 7
Token IDs: [9906, 11, 1268, 527, 499, 3432, 30]
Tokens: ['Hello', ',', ' how', ' are', ' you', ' today', '?']
Characters per token: 3.57

2️⃣ Technical/programming text:
Text: "def calculate_total(items): return sum(items)"
Token count: 9
Token IDs: [755, 11294, 11017, 25331, 1680, 471, 2694, 25331, 8]
Tokens: ['def', ' calculate', '_total', '(items', '):', ' return', ' sum', '(items', ')']
Characters per token: 5.00

3️⃣ Numbers and special characters:
Text: "Price: $1,234.56 (20% off!)"
Token count: 13
Token IDs: [7117, 25, 400, 16, 11, 11727, 13, 3487, 320, 508, 4, 1022, 16715]
Tokens: ['Price', ':', ' $', '1', ',', '234', '.', '56', ' (', '20', '%', ' off', '!)']
Characters per token: 2.08



[7117, 25, 400, 16, 11, 11727, 13, 3487, 320, 508, 4, 1022, 16715]

In [9]:
# Token efficiency varies by language and content type
print("=" * 70)
print("TOKENIZATION ACROSS LANGUAGES & CONTENT TYPES")
print("=" * 70)

test_texts = {
    "English": "Artificial intelligence is transforming the world.",
    "Hindi": "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।",
    "Spanish": "La inteligencia artificial está transformando el mundo.",
    "Code (Python)": "for i in range(10): print(f'Value: {i}')",
    "JSON": '{"name": "John", "age": 30, "city": "NYC"}',
    "Emoji-heavy": "🎉🚀 Amazing product launch! 💯🔥 #Success",
}

print("\nToken Efficiency Comparison:\n")
print(f"{'Content Type':<20} | {'Characters':>10} | {'Tokens':>7} | {'Chars/Token':>11}")
print("-" * 60)

for content_type, text in test_texts.items():
    tokens = encoding.encode(text)
    chars_per_token = len(text) / len(tokens)
    print(f"{content_type:<20} | {len(text):>10} | {len(tokens):>7} | {chars_per_token:>11.2f}")

print("\n💡 Key Insight: Non-English text and special characters often")
print("   require more tokens for the same semantic content!")

TOKENIZATION ACROSS LANGUAGES & CONTENT TYPES

Token Efficiency Comparison:

Content Type         | Characters |  Tokens | Chars/Token
------------------------------------------------------------
English              |         50 |       8 |        6.25
Hindi                |         41 |      43 |        0.95
Spanish              |         55 |      10 |        5.50
Code (Python)        |         40 |      15 |        2.67
JSON                 |         42 |      19 |        2.21
Emoji-heavy          |         38 |      17 |        2.24

💡 Key Insight: Non-English text and special characters often
   require more tokens for the same semantic content!


In [10]:
# Understanding how common words vs rare words tokenize
print("=" * 70)
print("COMMON vs RARE WORD TOKENIZATION")
print("=" * 70)

word_examples = [
    ("the", "Most common English word"),
    ("hello", "Common greeting"),
    ("artificial", "Common technical term"),
    ("tokenization", "Technical jargon"),
    ("pneumonoultramicroscopicsilicovolcanoconiosis", "Very rare/long word"),
    ("ChatGPT", "Brand name / neologism"),
    ("LangChain", "Technical tool name"),
]

print("\nHow different words get tokenized:\n")
for word, description in word_examples:
    tokens = encoding.encode(word)
    token_strings = [encoding.decode([t]) for t in tokens]
    print(f"'{word}' ({description})")
    print(f"  → {len(tokens)} token(s): {token_strings}")
    print()

print("💡 Key Insight: Common words are single tokens, while rare/long words")
print("   get split into multiple subword tokens!")

COMMON vs RARE WORD TOKENIZATION

How different words get tokenized:

'the' (Most common English word)
  → 1 token(s): ['the']

'hello' (Common greeting)
  → 1 token(s): ['hello']

'artificial' (Common technical term)
  → 2 token(s): ['art', 'ificial']

'tokenization' (Technical jargon)
  → 2 token(s): ['token', 'ization']

'pneumonoultramicroscopicsilicovolcanoconiosis' (Very rare/long word)
  → 17 token(s): ['p', 'ne', 'um', 'on', 'oul', 'tram', 'icro', 'sc', 'op', 'ics', 'il', 'ic', 'ov', 'ol', 'cano', 'con', 'iosis']

'ChatGPT' (Brand name / neologism)
  → 3 token(s): ['Chat', 'G', 'PT']

'LangChain' (Technical tool name)
  → 2 token(s): ['Lang', 'Chain']

💡 Key Insight: Common words are single tokens, while rare/long words
   get split into multiple subword tokens!


In [12]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=f"Explain AI Agents in 10 words",       
)

print('Response ', response.text)

print('Response Token Count ', response.usage_metadata.candidates_token_count)
print('Prompt Token Count ', response.usage_metadata.prompt_token_count)
print('Thinking Token Count ', response.usage_metadata.thoughts_token_count)
print('Total Token Count ', response.usage_metadata.total_token_count)

Response  Autonomous programs perceiving, reasoning, and acting to achieve goals.

Response Token Count  13
Prompt Token Count  8
Thinking Token Count  None
Total Token Count  21


---

## 3. Embeddings: Semantic Understanding

### What are Embeddings?

Embeddings convert text into numerical vectors that capture semantic meaning:
- Similar meanings → similar vectors (close in vector space)
- Used for: semantic search, clustering, recommendations

### Industry Example: Customer Inquiry Routing

Let's build a simple system that routes customer inquiries to the right department.

In [13]:
import numpy as np

# Department descriptions for routing
departments = {
    "billing": "Payment issues, invoices, refunds, subscription charges, billing disputes",
    "technical": "Software bugs, technical problems, system errors, integration issues, API",
    "sales": "Pricing questions, product information, upgrade plans, enterprise deals",
    "shipping": "Order status, delivery tracking, shipping delays, lost packages"
}

# Customer inquiries to route
inquiries = [
    "Why was I charged twice this month?",
    "The dashboard keeps crashing when I export data",
    "What's included in the Enterprise plan?",
    "My package hasn't arrived yet, it's been 2 weeks",
    "How do I connect your API to Salesforce?",
    "Can I get a refund for last month?"
]

def get_embedding(text: str) -> list:
    """Get embedding using Gemini's embedding model."""

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[text],
    )

    return result.embeddings[0].values

def cosine_similarity(a: list, b: list) -> float:
    """Calculate cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [14]:
billing_embeddings = get_embedding(departments['billing'])
billing_embeddings[:10]

[-0.0048097055,
 -0.007116501,
 0.018125683,
 -0.04232401,
 -0.017497873,
 -0.0023789566,
 0.039895453,
 0.017001031,
 -0.018179499,
 -0.011741438]

In [15]:
len(billing_embeddings)

3072

In [16]:
for inquery in inquiries:
    embeddings = get_embedding(inquery)
    print(f"Consine similarity with billing dept for inquiry '{inquery}': {cosine_similarity(embeddings, billing_embeddings):.2f}")

Consine similarity with billing dept for inquiry 'Why was I charged twice this month?': 0.59
Consine similarity with billing dept for inquiry 'The dashboard keeps crashing when I export data': 0.50
Consine similarity with billing dept for inquiry 'What's included in the Enterprise plan?': 0.51
Consine similarity with billing dept for inquiry 'My package hasn't arrived yet, it's been 2 weeks': 0.52
Consine similarity with billing dept for inquiry 'How do I connect your API to Salesforce?': 0.50
Consine similarity with billing dept for inquiry 'Can I get a refund for last month?': 0.61


In [17]:
print("CUSTOMER INQUIRY ROUTING SYSTEM")
print("="*60)

# Get embeddings for department descriptions
dept_embeddings = {dept: get_embedding(desc) for dept, desc in departments.items()}
len(dept_embeddings)

CUSTOMER INQUIRY ROUTING SYSTEM


4

In [18]:
dept_embeddings['billing'][:10]

[-0.0048097055,
 -0.007116501,
 0.018125683,
 -0.04232401,
 -0.017497873,
 -0.0023789566,
 0.039895453,
 0.017001031,
 -0.018179499,
 -0.011741438]

In [19]:
# Route each inquiry
print("\nRouting customer inquiries...\n")

for inquiry in inquiries:
    inquiry_embedding = get_embedding(inquiry)
    
    # Find most similar department
    scores = {
        dept: cosine_similarity(inquiry_embedding, emb)
        for dept, emb in dept_embeddings.items()
    }
    
    best_dept = max(scores, key=scores.get)
    confidence = scores[best_dept]
    
    print(f"Inquiry: \"{inquiry[:45]}...\"")
    print(f"  → Route to: {best_dept.upper()} (confidence: {confidence:.2f})")
    print()


Routing customer inquiries...

Inquiry: "Why was I charged twice this month?..."
  → Route to: BILLING (confidence: 0.59)

Inquiry: "The dashboard keeps crashing when I export da..."
  → Route to: TECHNICAL (confidence: 0.57)

Inquiry: "What's included in the Enterprise plan?..."
  → Route to: SALES (confidence: 0.67)

Inquiry: "My package hasn't arrived yet, it's been 2 we..."
  → Route to: SHIPPING (confidence: 0.67)

Inquiry: "How do I connect your API to Salesforce?..."
  → Route to: TECHNICAL (confidence: 0.56)

Inquiry: "Can I get a refund for last month?..."
  → Route to: BILLING (confidence: 0.61)



In [22]:
# Using LangChain embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Compare semantic similarity
texts = [
    "The product quality is excellent",
    "Great quality, love this item",
    "The shipping was very slow",
    "Delivery took forever"
]

print("SEMANTIC SIMILARITY DEMO (LangChain)")
print("="*60)

# Get all embeddings
all_embeddings = embeddings_model.embed_documents(texts)

# Create similarity matrix
print("\nSimilarity Matrix:")
print(f"{'':30} | " + " | ".join([f"Text {i+1}" for i in range(len(texts))]))
print("-" * 70)

for i, text in enumerate(texts):
    row = f"{text[:28]:30} | "
    similarities = []
    for j in range(len(texts)):
        sim = cosine_similarity(all_embeddings[i], all_embeddings[j])
        similarities.append(f"{sim:.2f}")
    print(row + "  |  ".join(similarities))

print("\nNotice: Similar meanings (1&2, 3&4) have high similarity scores!")

SEMANTIC SIMILARITY DEMO (LangChain)

Similarity Matrix:
                               | Text 1 | Text 2 | Text 3 | Text 4
----------------------------------------------------------------------
The product quality is excel   | 1.00  |  0.94  |  0.85  |  0.84
Great quality, love this ite   | 0.94  |  1.00  |  0.84  |  0.83
The shipping was very slow     | 0.85  |  0.84  |  1.00  |  0.96
Delivery took forever          | 0.84  |  0.83  |  0.96  |  1.00

Notice: Similar meanings (1&2, 3&4) have high similarity scores!


---

## 4. Streaming: Token-by-Token Generation

### Why Streaming Matters

LLMs generate text one token at a time (autoregressive generation). Streaming:
- Improves perceived latency (users see output immediately)
- Essential for real-time applications
- Shows the model "thinking" process

### Industry Example: Real-time Report Generation

In [24]:
import time

prompt = """
Generate a brief quarterly sales report summary for a SaaS company:
- Q4 Revenue: $2.5M (up 15% YoY)
- New Customers: 150
- Churn Rate: 3.2%
- Top Product: Enterprise Plan
"""

print("STREAMING GENERATION DEMO")
print("="*60)
print("Watch the report generate in real-time...\n")

response_stream = client.models.generate_content_stream(
    model="gemini-2.0-flash",
    contents=prompt       
)  

for chunk in response_stream:
    if chunk.text:
        print(chunk.text)
        # print(chunk.text, end="", flush=True)
        time.sleep(0.5)  # Small delay for visibility

print("\n\n" + "="*60)
print("Generation complete!")

STREAMING GENERATION DEMO
Watch the report generate in real-time...

##
 Q4 2023 Sales Report Summary

Q4 was a strong quarter for
 [SaaS Company Name], highlighted by significant revenue growth. Revenue reached **$2.5
M, a 15% increase year-over-year**. We acquired **150 new customers** and maintained a healthy churn rate of **3
.2%**. The **Enterprise Plan** continues to be our top-performing product, driving significant revenue contribution.



Generation complete!


In [26]:
# LangChain streaming
print("LANGCHAIN STREAMING")
print("="*60)

llm_stream = ChatGoogleGenerativeAI(model="gemini-2.0-flash", streaming=True)

print("Streaming response:\n")
for chunk in llm_stream.stream("Write a 3-sentence product launch announcement for an AI-powered CRM."):
    # print(chunk.content, end="", flush=True)
    print(chunk.content)
    time.sleep(0.5) 
print()

LANGCHAIN STREAMING
Streaming response:

We
're thrilled
 to announce the launch of "SynergyAI," the CRM that anticipates your customer
's needs before they even know them! Powered by cutting-edge AI, Synergy
AI automates tasks, personalizes interactions, and provides actionable insights to boost sales and strengthen customer relationships. Experience the future of CRM and unlock unprecedented growth with SynergyAI today
!





---

## 5. Temperature: Controlling Creativity vs Consistency

### What is Temperature?

Temperature controls the "randomness" of token selection:

| Temperature | Behavior | Use Case |
|-------------|----------|----------|
| 0.0 | Deterministic, always picks most likely token | Data extraction, classification |
| 0.3-0.5 | Mostly consistent with slight variation | Customer support, Q&A |
| 0.7-0.9 | Balanced creativity | General content, marketing copy |
| 1.0+ | High creativity, more random | Creative writing, brainstorming |

### Industry Example: Marketing Campaign Ideas

In [28]:
prompt = "Generate a single creative tagline for a sustainable fashion brand targeting millennials, just give the tagline:"

temperatures = [0.0, 0.5, 1.0, 1.5]

print("TEMPERATURE EFFECTS ON CREATIVITY")
print("="*60)
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"\n🌡️ Temperature: {temp}")
    print("-"*40)
    
    # Generate 3 samples at each temperature
    for i in range(3):

        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                max_output_tokens=50,
                temperature=temp
            ),        
        )        
        print(f"  {i+1}. {response.text.strip()[:60]}")

TEMPERATURE EFFECTS ON CREATIVITY
Prompt: Generate a single creative tagline for a sustainable fashion brand targeting millennials, just give the tagline:


🌡️ Temperature: 0.0
----------------------------------------
  1. Dress Good, Do Good.
  2. Dress Good, Do Good.
  3. Dress Good, Do Good.

🌡️ Temperature: 0.5
----------------------------------------
  1. Dress Good, Feel Good, Do Good.
  2. **Dress Good, Do Good.**
  3. Dress Good, Do Good.

🌡️ Temperature: 1.0
----------------------------------------
  1. **Dress Well, Live Consciously.**
  2. Dress well, Do Good.
  3. Dress good, do good.

🌡️ Temperature: 1.5
----------------------------------------
  1. Style Consciously. Live Beautifully.
  2. **Dress Good, Do Good.**
  3. Wear the Change.


In [30]:
# Using LangChain - comparing temperatures
from langchain_core.prompts import ChatPromptTemplate

brand = "eco-friendly sneakers made from recycled ocean plastic"
prompt = f"You are a creative marketing expert specializing in sustainable brands. Generate a tagline for: {brand}, provide just the tagline."

print("LANGCHAIN TEMPERATURE COMPARISON")
print("="*60)

for temp in [0.0, 0.7, 1.2]:
    llm_temp = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=temp)
    # chain = template | llm_temp
    
    
    print(f"\nTemperature {temp}:")
    for i in range(2):
        result = llm_temp.invoke(prompt)
        print(f"  {i+1}. {result.content[:60]}...")

LANGCHAIN TEMPERATURE COMPARISON

Temperature 0.0:
  1. Walk Lightly. Tread Responsibly....
  2. Walk Lightly. Tread Responsibly....

Temperature 0.7:
  1. Walk Lightly. Live Deeply....
  2. **Walk Lightly. Tread Responsibly.**...

Temperature 1.2:
  1. Walk Lighter. Live Brighter....
  2. **Walk Lightly. Live Deeply.**...


---

## 6. Max Tokens & Output Control

### Why Max Tokens Matters

- **Cost control**: Limits spending on long responses
- **Prevents runaway generation**: Model won't write forever
- **Application design**: Different use cases need different lengths

### Industry Example: Multi-length Content Generation

In [32]:
product_info = """
Product: SmartFit Pro Fitness Watch
Features: Heart rate monitoring, GPS tracking, 7-day battery, water resistant, 
sleep analysis, 50+ workout modes, smartphone notifications
Price: $199
Target: Health-conscious professionals aged 25-45
"""

content_types = {
    "tweet": 100,          # ~280 characters
    "email_subject": 50,   # Very short
    "product_summary": 100, # Brief paragraph
    "full_description": 300 # Detailed copy
}

print("MULTI-LENGTH CONTENT GENERATION")
print("="*60)

for content_type, max_tokens in content_types.items():
    # prompt = f"""
    # Based on this product info, write a {content_type.replace('_', ' ')}:
    # {product_info}, 
    # """
    prompt = f"""
    Based on this product info, write a {content_type.replace('_', ' ')}:
    {product_info}, 
    #generate within {int(max_tokens * 0.75)} tokens.
    """


    response = client.models.generate_content(
        model="models/gemini-3-flash-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
            max_output_tokens=max_tokens,
            temperature=0.7
        ),        
    )   

    
    print(f"\n📝 {content_type.upper()} (max {max_tokens} tokens):")
    print("-"*40)
    print(response.text)

MULTI-LENGTH CONTENT GENERATION

📝 TWEET (max 100 tokens):
----------------------------------------
Upgrade your hustle with the SmartFit Pro. ⌚✨

From the boardroom to the gym, track 50+ workouts, monitor heart rate, and analyze sleep—all with a 7-day battery. Built for the driven professional. 

🚀 Stay connected. Stay healthy. 

Shop now for $199! #SmartFitPro #FitnessGoals #TechStyle

📝 EMAIL_SUBJECT (max 50 tokens):
----------------------------------------
Master Your Day: SmartFit Pro for Professionals – Only $199!

📝 PRODUCT_SUMMARY (max 100 tokens):
----------------------------------------
The SmartFit Pro Fitness Watch ($199) is the ultimate companion for health-conscious professionals aged 25-45. It features advanced heart rate monitoring, GPS, sleep analysis, and 50+ workout modes to optimize your wellness. Designed for busy lifestyles, it offers smartphone notifications, a 7-day battery, and water resistance. Stay connected and track your goals effortlessly with this sleek, 

---

## 7. Practical Exercise: Build a Smart Content Analyzer

Let's combine everything we learned to build a practical content analysis tool.

In [ ]:
class ContentAnalyzer:
    """
    A practical content analyzer demonstrating LLM concepts:
    - Token estimation for cost planning
    - Temperature selection based on task
    - Embeddings for content categorization
    """
    
    def __init__(self):
        self.client =  genai.Client()
        self.model_name = "models/gemini-3-flash-preview"
        self.categories = {
            "product_review": "Customer opinions about product quality, features, and value",
            "support_request": "Technical issues, bugs, problems requiring assistance",
            "feature_request": "Suggestions for new features or improvements",
            "general_inquiry": "Questions about pricing, availability, or general information"
        }
        self._init_category_embeddings()
    
    def _init_category_embeddings(self):
        """Pre-compute category embeddings for fast classification."""
        self.category_embeddings = {}
        for cat, desc in self.categories.items():
      
            result = self.client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=[desc],
                    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
                )

            self.category_embeddings[cat] = result.embeddings[0].values
    
    def estimate_cost(self, text: str, model_price_per_million: float = 0.50) -> dict:
        """Estimate token count and cost."""
        import tiktoken
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens = len(encoding.encode(text))
        cost = (tokens / 1_000_000) * model_price_per_million
        return {
            "text_length": len(text),
            "estimated_tokens": tokens,
            "estimated_cost": f"${cost:.6f}"
        }
    
    def categorize(self, text: str) -> dict:
        """Categorize content using embeddings."""
        result = self.client.models.embed_content(
            model="gemini-embedding-001",
            contents=[text],
            config=types.EmbedContentConfig(task_type="retrieval_query")
        )
        text_embedding = result.embeddings[0].values
        
        scores = {}
        for cat, emb in self.category_embeddings.items():
            scores[cat] = cosine_similarity(text_embedding, emb)
        
        best_category = max(scores, key=scores.get)
        return {
            "category": best_category,
            "confidence": scores[best_category],
            "all_scores": scores
        }
    
    def analyze_sentiment(self, text: str) -> str:
        """Analyze sentiment with low temperature for consistency."""

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=f"Classify sentiment as positive, negative, or neutral: {text}",
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
                max_output_tokens=10,
                temperature=0
            ),        
        )   

        return response.text.strip().lower()
    
    def generate_response(self, text: str, style: str = "professional") -> str:
        """Generate response with appropriate temperature."""
        temp_map = {
            "professional": 0.3,
            "friendly": 0.7,
            "creative": 1.0
        }

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=f"Write  one  response with style {style} to this customer message, share only the response with 100 words: {text}",
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
                max_output_tokens=150,
                temperature=temp_map.get(style, 0.5)
            ),        
        )   

        return response.text.strip()
    
    def full_analysis(self, text: str) -> dict:
        """Complete analysis of content."""
        return {
            "cost_estimate": self.estimate_cost(text),
            "category": self.categorize(text),
            "sentiment": self.analyze_sentiment(text),
            "suggested_response": self.generate_response(text)
        }

# Demo the analyzer
analyzer = ContentAnalyzer()

test_messages = [
    "Your product is amazing! The quality exceeded my expectations.",
    "The app keeps crashing every time I try to upload a photo.",
    "It would be great if you could add dark mode to the mobile app.",
]

print("CONTENT ANALYZER DEMO")
print("="*70)

for msg in test_messages:
    print(f"\n📧 Message: \"{msg}\"")
    print("-"*60)
    
    analysis = analyzer.full_analysis(msg)
    
    print(f"📊 Category: {analysis['category']['category']} ({analysis['category']['confidence']:.2f})")
    print(f"😊 Sentiment: {analysis['sentiment']}")
    print(f"💰 Est. Tokens: {analysis['cost_estimate']['estimated_tokens']}")
    print(f"\n💬 Suggested Response:")
    print(f"   {analysis['suggested_response']}...")

---

## Key Takeaways

### 1. Tokens are Currency
- Understand token counting for cost optimization
- Different content tokenizes differently
- Concise prompts save money at scale

### 2. Temperature Controls Creativity
- Low (0-0.3): Classification, extraction, factual Q&A
- Medium (0.5-0.7): General tasks, customer support
- High (0.9+): Creative writing, brainstorming

### 3. Embeddings Enable Understanding
- Semantic search and routing
- Content categorization
- Finding similar examples

### 4. Streaming Improves UX
- Users see output immediately
- Essential for production applications

---

## Next Steps

Now that you understand how LLMs work, proceed to:
- **Lesson 2.1**: Prompting Fundamentals - Master the anatomy of effective prompts